# Step 2 — Masking of employer, contact and source fields

**Input:** the Step 1 outputs (`step1_modeling_fields.csv`, `step1_restricted_metadata.csv`).
**Output:** a masked modeling text in which the job title (b) and qualification block (c) are retained verbatim, and the employer name (a), contact/application block (d) and source URL (e) are masked.

We adopted the masking approach of Dikshit et al. (2024) with our own modifications. We masked the employer name, the contact/application block and the source URL while retaining the original values as restricted metadata, to reduce the risk that a model recognizes employer-specific templates instead of wording patterns. The job title and the qualification block are the substantive wording under study and are therefore not altered, apart from any residual employer or contact string found inside them.

## 0. Setup and loading

The Step 1 files are joined on `record_id`. Nothing is re-parsed here; the field assignment made in Step 1 is taken as given. The two inputs are read as text (`dtype=str`) so that identifiers and phone-like strings are not coerced to numbers.

In [ ]:
from pathlib import Path
import hashlib
import re

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

STEP1_DIR = Path("outputs") / "equal_step1_structural_parsing"
OUTPUT_DIR = Path("outputs") / "equal_step2_masking"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

fields = pd.read_csv(STEP1_DIR / "step1_modeling_fields.csv", encoding="utf-8-sig", dtype=str).fillna("")
meta = pd.read_csv(STEP1_DIR / "step1_restricted_metadata.csv", encoding="utf-8-sig", dtype=str).fillna("")
data = fields.merge(meta, on="record_id", how="inner", validate="one_to_one")
assert len(data) == len(fields) == len(meta), "Step 1 files do not align on record_id"
print(f"{len(data):,} advertisements loaded from Step 1.")

## 1. What is masked and what is retained

| Step 1 field | Treatment | Placeholder in analysis data | Original value |
|---|---|---|---|
| (a) Employer name | **Masked** in the metadata and wherever the name (or its name without a legal suffix such as *Inc.*, *Corp.*) is repeated inside the retained text | `[EMPLOYER]` | kept in restricted metadata |
| (b) Job title | **Retained** | – | – |
| (c) Qualification block | **Retained** (only residual employer/contact strings are masked) | – | – |
| (d) Contact/application block | **Masked** as a whole field; any e-mail, URL or phone-like string that remains inside (b)/(c) is also masked | `[CONTACT]` (field); `[EMAIL]`, `[URL]`, `[PHONE]` (inline) | kept in restricted metadata |
| (e) Source URL | **Masked** | `[SOURCE_URL]` | kept in restricted metadata |

Text that Step 1 kept without a qualification header (`body_other`) stays in the analysis text, because in those ads it carries the qualification wording; it is masked with the same inline rules and is reported separately so it can be reviewed.

A pseudonymous `employer_id` (a short hash of the lower-cased employer name) is kept next to the masked text. It lets later steps group ads by employer — for example, to keep one employer's ads in the same train/test split or to report employer-level concentration — without exposing the name. It is derived from a name that can be guessed, so it should be treated as restricted too.

In [ ]:
def employer_variants(name):
    name = re.sub(r"\s+", " ", name).strip()
    if not name:
        return []
    core = re.sub(r"[,\s]+(?:inc|corp|corporation|incorporated|co|ltd|llc)\.?$", "", name, flags=re.I).strip(" ,.")
    variants = {name, core}
    return sorted((v for v in variants if len(v) >= 4), key=len, reverse=True)


def mask_employer(text, name):
    for v in employer_variants(name):
        pattern = r"(?<!\w)" + re.sub(r"\\\s+|\\ ", r"\\s+", re.escape(v)) + r"(?!\w)"
        text = re.sub(pattern, "[EMPLOYER]", text, flags=re.I)
    return text


EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w\-]+\.(?:com|ph|org|net|edu|gov)(?:/\S*)?\b", re.I)
PHONE_RE = re.compile(r"(?<!\d)(?:\+?63|0)?\d[\d\s\-()]{6,}\d(?!\d)")


def mask_inline_contact(text):
    text = EMAIL_RE.sub("[EMAIL]", text)
    text = URL_RE.sub("[URL]", text)
    return PHONE_RE.sub("[PHONE]", text)


def mask_text(text, employer):
    return mask_inline_contact(mask_employer(text, employer))


def employer_id(name):
    key = re.sub(r"\s+", " ", name).strip().lower()
    return "EMP-" + hashlib.sha256(key.encode("utf-8")).hexdigest()[:8] if key else ""

## 2. Applying the masks

The employer is masked first, then inline contact strings, so that an e-mail such as `hr@yamaha-motor.com.ph` is replaced as a whole. Masking is applied separately to the title, the body text and the qualification block; the three are then recombined in the same order as Step 1 to form `masked_text`. In the metadata columns the employer, the contact block and the source URL are replaced by their placeholders (an empty field stays empty, so missing values are not disguised as masked ones).

In [ ]:
data["masked_job_title"] = [mask_text(t, e) for t, e in zip(data["job_title_clean"], data["employer_name"])]
data["masked_body_other_text"] = [mask_text(t, e) for t, e in zip(data["body_other_text"], data["employer_name"])]
data["masked_qualification_text"] = [mask_text(t, e) for t, e in zip(data["qualification_text"], data["employer_name"])]
data["masked_text"] = data.apply(
    lambda r: "\n".join(x for x in [r["masked_job_title"], r["masked_body_other_text"], r["masked_qualification_text"]] if x),
    axis=1,
)

data["employer_id"] = data["employer_name"].map(employer_id)
data["employer_masked"] = data["employer_name"].map(lambda v: "[EMPLOYER]" if v else "")
data["contact_masked"] = data["contact_text"].map(lambda v: "[CONTACT]" if v else "")
data["source_masked"] = data["source_url"].map(lambda v: "[SOURCE_URL]" if v else "")
print("Masking complete.")

## 3. Validation

Three checks are reported. (i) **Employer leakage:** the employer name (or its suffix-free form) must not appear in `masked_text`. (ii) **Contact leakage:** no e-mail, URL or phone-like string may remain. (iii) **Retention:** the share of job titles left unchanged, and how many qualification blocks needed any inline masking; a large change in either would mean the masks are touching wording that should be preserved. Zero leakage shows the masks were applied consistently; it does not prove that an employer cannot be inferred from other wording (see Limitations).

In [ ]:
def leaks_employer(text, name):
    return any(re.search(r"(?<!\w)" + re.escape(v) + r"(?!\w)", text, re.I) for v in employer_variants(name))


emp_leak = [leaks_employer(t, e) for t, e in zip(data["masked_text"], data["employer_name"])]
contact_leak = data["masked_text"].map(lambda t: bool(EMAIL_RE.search(t) or URL_RE.search(t) or PHONE_RE.search(t)))
print(f"Ads with employer name left in masked_text : {sum(emp_leak):,}")
print(f"Ads with contact-like strings left          : {int(contact_leak.sum()):,}")

summary = pd.DataFrame(
    {
        "measure": [
            "ads with [EMPLOYER] inside retained text",
            "ads with [EMAIL]/[URL]/[PHONE] inside retained text",
            "job titles changed by masking",
            "qualification blocks changed by masking",
            "body_other texts changed by masking",
            "ads with employer masked in metadata",
            "ads with contact block masked",
            "ads with source URL masked",
        ],
        "ads": [
            int(data["masked_text"].str.contains("[EMPLOYER]", regex=False).sum()),
            int(data["masked_text"].str.contains(r"\[(?:EMAIL|URL|PHONE)\]").sum()),
            int((data["masked_job_title"] != data["job_title_clean"]).sum()),
            int((data["masked_qualification_text"] != data["qualification_text"]).sum()),
            int((data["masked_body_other_text"] != data["body_other_text"]).sum()),
            int((data["employer_masked"] != "").sum()),
            int((data["contact_masked"] != "").sum()),
            int((data["source_masked"] != "").sum()),
        ],
    }
)
summary["share"] = (summary["ads"] / len(data)).round(3)
display(summary)

# Titles that the employer mask touched, for reviewer attention.
display(
    data.loc[data["masked_job_title"] != data["job_title_clean"], ["record_id", "job_title_clean", "masked_job_title"]].head(10)
)

## 4. Side-by-side examples

A reproducible sample (seed 42) of ads in which the retained text was changed by masking, followed by one ad in which only the metadata fields were masked. The *original* column contains real employer and contact details, so treat saved outputs of this cell as restricted.

In [ ]:
def show(row):
    print("\n" + "=" * 90)
    print(f"{row['record_id']}  |  employer_id: {row['employer_id']}")
    print(f"(a) employer      : {row['employer_name']}   ->   {row['employer_masked']}")
    print(f"(b) job title     : {row['job_title_clean']}   ->   {row['masked_job_title']}")
    print("(c) qualification :")
    print("    ORIGINAL:\n" + (row["qualification_text"][:500] or "    -"))
    print("    MASKED:\n" + (row["masked_qualification_text"][:500] or "    -"))
    print(f"(d) contact block : {row['contact_text'][:80].replace(chr(10), ' / ')}...   ->   {row['contact_masked']}")
    print(f"(e) source URL    : {row['source_url'][:70]}   ->   {row['source_masked']}")


changed_text = data[(data["masked_text"] != data["job_title_clean"] + "\n" + data["body_other_text"] + "\n" + data["qualification_text"]) & (data["masked_text"].str.contains(r"\[(?:EMPLOYER|EMAIL|URL|PHONE)\]"))]
print(f"Ads whose retained text was changed by masking: {len(changed_text):,}")
if len(changed_text):
    for _, row in changed_text.sample(n=min(2, len(changed_text)), random_state=RANDOM_STATE).iterrows():
        show(row)
print("\n##### metadata-only masking (retained text unchanged)")
show(data[~data["record_id"].isin(changed_text["record_id"]) & (data["qualification_text"] != "")].sample(n=1, random_state=RANDOM_STATE).iloc[0])

## 5. Exports

The analysis file contains only masked text and the pseudonymous `employer_id`. The restricted file maps each `record_id` back to the original employer, contact block and source URL and must be stored separately with restricted access. Both are keyed by `record_id`.

In [ ]:
data[["record_id", "employer_id", "masked_job_title", "masked_body_other_text", "masked_qualification_text", "masked_text", "flags"]].to_csv(
    OUTPUT_DIR / "step2_masked_analysis_text.csv", index=False, encoding="utf-8-sig"
)
data[["record_id", "employer_id", "employer_name", "contact_text", "source_url"]].to_csv(
    OUTPUT_DIR / "step2_restricted_original_fields.csv", index=False, encoding="utf-8-sig"
)
print("Saved:", *sorted(p.name for p in OUTPUT_DIR.iterdir()), sep="\n  ")

## 6. Limitations

- Only the employer name and its suffix-free form are masked. Abbreviations, brand names, branch names or product names that are not the employer's recorded name are not caught.
- Repeated employer-specific *template lines* (for example, the same site-assignment sentence in many ads from one employer) are retained, because they belong to the title/qualification wording; they are not masked here and remain a possible employer cue for a model.
- Work-site addresses that Step 1 left in the body text are retained, so place names can still identify an employer indirectly.
- `employer_id` is a hash of a guessable name and is pseudonymous, not anonymous.
- Zero leakage in the validation means the masks were applied consistently, not that re-identification is impossible.